In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Youtube").getOrCreate()

In [7]:
df=spark.read.csv('../Data/youtube/USvideos.csv',header=True)

In [8]:
df.show()

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...|        CaseyNeistat|         22|2017-11-13T17:13:...|     SHANtell martin| 748374| 57527|    2966|        15954|https://i.ytimg.c...|            False|           Fal

In [ ]:
from pyspark.sql.functions import avg, sum, count,desc
df.groupBy(df["channel_title"]).agg(count('channel_title').alias('Num_Trending')).orderBy(desc('Num_Trending')).show()

+--------------------+------------+
|       channel_title|Num_Trending|
+--------------------+------------+
|                ESPN|         203|
|The Tonight Show ...|         197|
|             Netflix|         193|
|        TheEllenShow|         193|
|                 Vox|         193|
|The Late Show wit...|         187|
|   Jimmy Kimmel Live|         186|
|Late Night with S...|         183|
|      Screen Junkies|         182|
|                 NBA|         181|
|                 CNN|         180|
| Saturday Night Live|         175|
|               WIRED|         171|
|       BuzzFeedVideo|         169|
|             INSIDER|         167|
|The Late Late Sho...|         163|
|              TED-Ed|         162|
|           Tom Scott|         159|
|                 WWE|         157|
|        CollegeHumor|         156|
+--------------------+------------+
only showing top 20 rows



In [10]:
from pyspark.sql.functions import col, split, explode, avg, count, desc, expr, length, to_date, to_timestamp

df = df.withColumn("views", col("views").cast("int"))\
           .withColumn("likes", col("likes").cast("int"))\
           .withColumn("dislikes", col("dislikes").cast("int"))\
           .withColumn("comment_count", col("comment_count").cast("int"))\
           .withColumn("trending_date", to_date(col("trending_date"), "yy.dd.MM"))\
           .withColumn("publish_time", to_timestamp(col("publish_time")))
df.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: date (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: integer (nullable = true)
 |-- likes: integer (nullable = true)
 |-- dislikes: integer (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [11]:
from pyspark.sql.functions import  when
df.groupBy("video_id", "trending_date").count().filter("count > 1").show()


+--------------------+-------------+-----+
|            video_id|trending_date|count|
+--------------------+-------------+-----+
|\nMust-See WWE vi...|         NULL|  126|
|\nTwitter: https:...|         NULL|   14|
|\nAccess Hollywoo...|         NULL|   14|
|\nhttps://www.you...|         NULL|   14|
|\nInstagram: http...|         NULL|   14|
|           \nFashion|         NULL|   33|
|\n» SUBSCRIBE: ht...|         NULL|   14|
|\nFor more than 5...|         NULL|   49|
|\nCook with confi...|         NULL|  108|
|\nCheck out Liza ...|         NULL|    5|
|\nTumblr: http://...|         NULL|   14|
|\nWant even more?...|         NULL|   14|
|\nInstagram: http...|         NULL|   14|
|\nGoogle+: https:...|         NULL|   14|
|\nWeb: http://wir...|         NULL|   14|
|\nHailing from We...|         NULL|    2|
|                 \n |         NULL|  132|
| \nABOUT BON APPÉTIT|         NULL|  108|
| \nABOUT VANITY FAIR|         NULL|  116|
|       \nABOUT VOGUE|         NULL|  113|
+----------

In [15]:
df.select(count(when(col('title').isNull(),col('title'))).alias("title")).show()

+-----+
|title|
+-----+
|    0|
+-----+



In [16]:
df.select([count(when(c.isNull(),c)).alias(c) for c in df.columns]).show()

AttributeError: 'str' object has no attribute 'isNull'

In [17]:
df = df.withColumn("lag_days", expr("datediff(trending_date, publish_time)"))
df.select("title", "lag_days").orderBy("lag_days").show(5)

+-------+--------+
|  title|lag_days|
+-------+--------+
|   NULL|    NULL|
|   NULL|    NULL|
|   NULL|    NULL|
|   NULL|    NULL|
| videos|    NULL|
+-------+--------+
only showing top 5 rows



In [18]:
df.show()

+-----------+-------------+--------------------+--------------------+-----------+-------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+--------+
|   video_id|trending_date|               title|       channel_title|category_id|       publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|lag_days|
+-----------+-------------+--------------------+--------------------+-----------+-------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+--------+
|2kyS6SvSYSE|   2017-11-14|WE WANT TO TALK A...|        CaseyNeistat|         22|2017-11-13 22:43:01|     SHANtell martin| 748374| 57527|    2966|        15954|https://i.ytimg.c...|         

In [19]:
df_cleaned = df.dropna(subset=["title", "publish_time"])

In [21]:
df.count()


48137

In [22]:
df_cleaned.count()

40949

In [30]:
from pyspark.sql.functions import format_number

df=df.withColumn('like_percent',format_number((col('likes')/col('views'))*100,2))
df.select("Title","channel_title","like_percent").orderBy(desc("like_ratio")).show()

+-------------------------+------------------+------------+
|                    Title|     channel_title|like_percent|
+-------------------------+------------------+------------+
|     Bruno Mars - Fine...|        Bruno Mars|       29.05|
|     Luis Fonsi, Demi ...|     LuisFonsiVEVO|       27.06|
|     j-hope 'Airplane' MV|           ibighit|       26.57|
|     dodie - Secret Fo...|         dodieVEVO|       25.37|
|     Louis Tomlinson -...|LouisTomlinsonVEVO|       24.51|
|BTS (방탄소년단) 'FAKE...|           ibighit|       24.44|
|     5 Seconds Of Summ...|          5SOSVEVO|       24.26|
|     Shawn Mendes: The...|      Shawn Mendes|       24.07|
|     Shawn Mendes: The...|      Shawn Mendes|       23.06|
|     Harry Styles - Ki...|   HarryStylesVEVO|       22.85|
|     Idol EXPOSES Dark...|  Henry Prince Mak|       22.82|
|     j-hope 'Airplane' MV|           ibighit|       22.80|
|     Shawn Mendes: The...|      Shawn Mendes|       22.80|
|BTS (방탄소년단) 'MIC ...|           ibighit|    

In [31]:
days_trending = df.groupBy("video_id", "title").agg(count('trending_date').alias('Trending_days'))
days_trending.orderBy(desc("Trending_days")).show()

+-----------+----------------------------+-------------+
|   video_id|                       title|Trending_days|
+-----------+----------------------------+-------------+
|j4KvrAUjn6c|        WE MADE OUR MOM C...|           30|
|r-3iathMo7o|        The ULTIMATE $30,...|           29|
|QBL8IRJ5yHU|        Why I'm So Scared...|           29|
|NBSAQenU2Bk|        Rooster Teeth Ani...|           29|
|8h--kFui1JA|        Sam Smith - Pray ...|           29|
|t4pRQ0jn23Q|        YoungBoy Never Br...|           29|
|iILJvqrAQ_w|        Charlie Puth - BO...|           29|
|2PH7dK6SLC8|        John Mayer - New ...|           28|
|mdWcaWBxxcY|        Rita Ora - Girls ...|           28|
|ulNswX3If6U|        Selena Gomez - Ba...|           28|
|YI3tsmFsrOg|        The Deadliest Bei...|           28|
|WIV3xNz8NoM|          Cobra Kai Season 2|           28|
|MAjY8mCTXWk|周杰倫 Jay Chou【不愛我就...|           28|
|6S9c5nnDd_s|        Bohemian Rhapsody...|           28|
|vjSohj-Iclc|        Getting some air,.

In [45]:
from pyspark.sql.functions import replace,regexp_replace 

df=df.withColumn("tags_array",split(regexp_replace (col("tags"), "\"", ""),"\|"))
df.select("tags_array","tags").show()

+--------------------+--------------------+
|          tags_array|                tags|
+--------------------+--------------------+
|   [SHANtell martin]|     SHANtell martin|
|[last week tonigh...|"last week tonigh...|
|[racist superman,...|"racist superman"...|
|[rhett and link, ...|"rhett and link"|...|
|[ryan, higa, higa...|"ryan"|"higa"|"hi...|
|[ijustine, week w...|"ijustine"|"week ...|
|[SNL, Saturday Ni...|"SNL"|"Saturday N...|
|[5 Ice Cream Gadg...|"5 Ice Cream Gadg...|
|[Trailer, Hugh Ja...|"Trailer"|"Hugh J...|
|[vox.com, vox, ex...|"vox.com"|"vox"|"...|
|[NFL, Football, o...|"NFL"|"Football"|...|
|[The Walking Dead...|"The Walking Dead...|
|[marshmello, bloc...|"marshmello"|"blo...|
|[nowthis, nowthis...|"nowthis"|"nowthi...|
|[shopping for new...|"shopping for new...|
|[Robots, Boston D...|"Robots"|"Boston ...|
|[pacific rim, pac...|"pacific rim"|"pa...|
|[TED, TED-Ed, TED...|"TED"|"TED-Ed"|"T...|
|[ultralight, airp...|"ultralight"|"air...|
|[SciShow, science...|"SciShow"|

In [46]:
tags_df=df.select("video_id",explode("tags_array").alias("tag"))
tags_df.groupBy(col("tag")).agg(count('video_id').alias("Number_of_videos")).orderBy(desc('Number_of_videos')).show()

+---------+----------------+
|      tag|Number_of_videos|
+---------+----------------+
|    funny|            3603|
|   comedy|            2931|
|   how to|            1604|
|   [none]|            1535|
|    music|            1302|
|      Pop|            1280|
|     2018|            1275|
|    humor|            1185|
|     food|            1159|
|  science|            1111|
|   review|            1005|
|   makeup|             990|
|     news|             988|
|celebrity|             930|
|     vlog|             928|
|    video|             890|
| tutorial|             864|
|     live|             862|
| comedian|             861|
|interview|             845|
+---------+----------------+
only showing top 20 rows



In [48]:
length_df=df.withColumn('title_length',length(col("title")))
length_df.select('title','title_length','views').orderBy(desc("views")).show()

+--------------------+------------+---------+
|               title|title_length|    views|
+--------------------+------------+---------+
|Childish Gambino ...|          51|225211923|
|Childish Gambino ...|          51|220490543|
|Childish Gambino ...|          51|217750076|
|Childish Gambino ...|          51|210338856|
|Childish Gambino ...|          51|205643016|
|Childish Gambino ...|          51|200820941|
|Childish Gambino ...|          51|196222618|
|Childish Gambino ...|          51|190950401|
|Childish Gambino ...|          51|184446490|
|Childish Gambino ...|          51|179045286|
|Childish Gambino ...|          51|173478072|
|Childish Gambino ...|          51|167997997|
|Childish Gambino ...|          51|162556776|
|Childish Gambino ...|          51|156612892|
|Childish Gambino ...|          51|149830680|
|YouTube Rewind: T...|          50|149376127|
|Ariana Grande - N...|          36|148689896|
|Childish Gambino ...|          51|142421830|
|Becky G, Natti Na...|          52

In [56]:
length_df.select('title','title_length','views').orderBy(desc("views")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [views#165 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(views#165 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=1254]
      +- Project [title#19, length(title#19) AS title_length#1246, cast(views#24 as int) AS views#165]
         +- FileScan csv [title#19,views#24] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/y:/DEVELOPER.GIT/pyspark/Data/youtube/USvideos.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<title:string,views:string>




In [49]:
popular_channels = df.groupBy("channel_title").agg(count("video_id").alias("trending_count"))
popular_channels.orderBy(desc("trending_count")).show(10)

+--------------------+--------------+
|       channel_title|trending_count|
+--------------------+--------------+
|                NULL|          6025|
|                ESPN|           203|
|The Tonight Show ...|           197|
|        TheEllenShow|           193|
|             Netflix|           193|
|                 Vox|           193|
|The Late Show wit...|           187|
|   Jimmy Kimmel Live|           186|
|Late Night with S...|           183|
|      Screen Junkies|           182|
+--------------------+--------------+
only showing top 10 rows



In [50]:
one_hit = popular_channels.filter("trending_count = 1")
one_hit.show(10)

+--------------------+--------------+
|       channel_title|trending_count|
+--------------------+--------------+
|     90s Commercials|             1|
|         Mayo Clinic|             1|
| video and featur...|             1|
|          BTS Videos|             1|
|               Ozuna|             1|
|           chris lee|             1|
|     BrianJustinCrum|             1|
|Guinness World Re...|             1|
|             Hot Dad|             1|
|           TrainVEVO|             1|
+--------------------+--------------+
only showing top 10 rows



In [53]:
video = df.filter(col("title").like("Childish%"))
video.groupBy("trending_date").agg(avg("views"),count('views')).orderBy("trending_date").show()

+-------------+------------+------------+
|trending_date|  avg(views)|count(views)|
+-------------+------------+------------+
|   2018-01-31|     33977.0|           1|
|   2018-02-01|     41187.0|           1|
|   2018-02-02|     45456.0|           1|
|   2018-02-03|     49947.0|           1|
|   2018-05-08| 1.6505018E7|           2|
|   2018-05-09|2.45767465E7|           2|
|   2018-05-10|3.15632455E7|           2|
|   2018-05-11|3.80147305E7|           2|
|   2018-05-12|4.39646115E7|           2|
|   2018-05-13| 5.1021397E7|           2|
|   2018-05-15|1.16581406E8|           1|
|   2018-05-16|1.26191952E8|           1|
|   2018-05-17|1.34839555E8|           1|
|   2018-05-18| 1.4242183E8|           1|
|   2018-05-19| 1.4983068E8|           1|
|   2018-05-20|1.56612892E8|           1|
|   2018-05-21|1.62556776E8|           1|
|   2018-05-22|1.67997997E8|           1|
|   2018-05-23|1.73478072E8|           1|
|   2018-05-24|1.79045286E8|           1|
+-------------+------------+------

In [55]:
df.filter(col("title").like("Childish%")).explain(True)

== Parsed Logical Plan ==
'Filter 'title LIKE Childish%
+- Project [video_id#17, trending_date#233, title#19, channel_title#20, category_id#21, publish_time#250, tags#23, views#165, likes#182, dislikes#199, comment_count#216, thumbnail_link#28, comments_disabled#29, ratings_disabled#30, video_error_or_removed#31, description#32, lag_days#336, like_ratio#600, like_percent#699, split(regexp_replace(tags#23, ", , 1), \|, -1) AS tags_array#1154]
   +- Project [video_id#17, trending_date#233, title#19, channel_title#20, category_id#21, publish_time#250, tags#23, views#165, likes#182, dislikes#199, comment_count#216, thumbnail_link#28, comments_disabled#29, ratings_disabled#30, video_error_or_removed#31, description#32, lag_days#336, like_ratio#600, like_percent#699, split(tags#23, \|, -1) AS tags_array#1121]
      +- Project [video_id#17, trending_date#233, title#19, channel_title#20, category_id#21, publish_time#250, tags#23, views#165, likes#182, dislikes#199, comment_count#216, thumbnail

In [ ]:
all_titles = df.select("title").rdd.flatMap(lambda x: x).collect()
with open("/tmp/titles.txt", "w") as f:
    f.write(" ".join(all_titles))

In [54]:
days_trending.write.option("header", True).csv("/output/days_trending")

Py4JJavaError: An error occurred while calling o623.csv.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:793)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1249)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1454)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:601)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:761)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.getAllCommittedTaskPaths(FileOu